In [2]:
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
import sys
import requests
from io import BytesIO
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from PIL import Image
import os
import numpy as np

sys.path.append('..')
from wc26_data import get_wc26_data, get_country_rankings, get_ids

/Users/chancetokubo/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
matches = get_ids('/Users/chancetokubo/bad.sports.metrics/matches.txt')
data = get_wc26_data(matches=matches)

Approximate time of data pulling completion: 144 seconds.
Dataframe Complete


In [4]:
rankings = get_country_rankings()

Dataframe complete.


In [7]:
data.head()

,Date,Group,Stage,TeamID,TeamName,TeamAbbr,Formation,IdMatch,GameNum,StatsId,...,SubstitutionsIn,SubstitutionsOut,TakeOnsCompleted,Threat,ThrowIns,TimePlayed,TopSpeed,TotalDistance,XG,YellowCards
0,2026-06-11T19:00:00Z,Group A,First Stage,43911,Mexico,MEX,4-1-2-3,400021443,1,151600,...,5.0,5.0,7.0,65.0,21.0,101.004683,33.307205,107310.437500,1.779373,1.0
1,2026-06-11T19:00:00Z,Group A,First Stage,43883,South Africa,RSA,5-3-2,400021443,1,151600,...,4.0,4.0,0.0,35.0,11.0,101.004683,34.370171,97060.179688,0.103336,2.0
2,2026-06-12T02:00:00Z,Group A,First Stage,43822,Korea Republic,KOR,3-4-3,400021441,2,151608,...,5.0,5.0,11.0,55.0,26.0,99.607383,35.170654,111805.359375,1.769696,1.0
3,2026-06-12T02:00:00Z,Group A,First Stage,43995,Czechia,CZE,5-2-3,400021441,2,151608,...,4.0,4.0,4.0,45.0,16.0,99.607383,33.999153,117628.406250,0.636732,0.0
4,2026-06-12T19:00:00Z,Group B,First Stage,43899,Canada,CAN,4-4-2,400021449,3,151614,...,5.0,5.0,8.0,68.0,32.0,100.330700,34.619015,111338.789062,1.494709,2.0


In [8]:
data = data.merge(rankings[['IdCountry', 'TotalPoints']], left_on='TeamAbbr', right_on='IdCountry', how='left').drop(columns='IdCountry')

opponent_elo = data.groupby('IdMatch')['TotalPoints'].transform(lambda x: x.iloc[::-1].values)
data['OpponentELO'] = opponent_elo

In [9]:
data.head()

,Date,Group,Stage,TeamID,TeamName,TeamAbbr,Formation,IdMatch,GameNum,StatsId,...,TakeOnsCompleted,Threat,ThrowIns,TimePlayed,TopSpeed,TotalDistance,XG,YellowCards,TotalPoints,OpponentELO
0,2026-06-11T19:00:00Z,Group A,First Stage,43911,Mexico,MEX,4-1-2-3,400021443,1,151600,...,7.0,65.0,21.0,101.004683,33.307205,107310.437500,1.779373,1.0,1700.984519,1418.211807
1,2026-06-11T19:00:00Z,Group A,First Stage,43883,South Africa,RSA,5-3-2,400021443,1,151600,...,0.0,35.0,11.0,101.004683,34.370171,97060.179688,0.103336,2.0,1418.211807,1700.984519
2,2026-06-12T02:00:00Z,Group A,First Stage,43822,Korea Republic,KOR,3-4-3,400021441,2,151608,...,11.0,55.0,26.0,99.607383,35.170654,111805.359375,1.769696,1.0,1612.547459,1481.485998
3,2026-06-12T02:00:00Z,Group A,First Stage,43995,Czechia,CZE,5-2-3,400021441,2,151608,...,4.0,45.0,16.0,99.607383,33.999153,117628.406250,0.636732,0.0,1481.485998,1612.547459
4,2026-06-12T19:00:00Z,Group B,First Stage,43899,Canada,CAN,4-4-2,400021449,3,151614,...,8.0,68.0,32.0,100.330700,34.619015,111338.789062,1.494709,2.0,1551.504089,1406.176640


In [ ]:
#pressure using finalthirdpitch control, possession, phaseaggregatefinalthird

In [38]:
mydata = data[['TeamName','TeamAbbr','IdMatch','TotalPoints','OpponentELO','FinalThirdPitchControl','Possession','PhaseAggregateFinalThird','XG','Goals']]

In [46]:
GoalvEGoal = mydata['Goals'] - mydata['XG']
mydata['GvEG'] = GoalvEGoal
mydata['GvEGS'] = GoalvEGoal / mydata['XG']

/var/folders/vl/tq28lccn1d19hq22zfhh94lc0000gn/T/ipykernel_68865/702340752.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mydata['GvEG'] = GoalvEGoal
/var/folders/vl/tq28lccn1d19hq22zfhh94lc0000gn/T/ipykernel_68865/702340752.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mydata['GvEGS'] = GoalvEGoal / mydata['XG']


In [48]:
GPMinFinalThird = mydata['Goals'] / mydata['PhaseAggregateFinalThird']
mydata['GPMFT'] = GPMinFinalThird

/var/folders/vl/tq28lccn1d19hq22zfhh94lc0000gn/T/ipykernel_68865/232542093.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mydata['GPMFT'] = GPMinFinalThird


In [49]:
mydata

,TeamName,TeamAbbr,IdMatch,TotalPoints,OpponentELO,FinalThirdPitchControl,Possession,PhaseAggregateFinalThird,XG,Goals,GvEG,GvEGS,GPMFT
0,Mexico,MEX,400021443,1700.984519,1418.211807,62.0,0.570936,10.756574,1.779373,2.0,0.220627,0.123991,0.185933
1,South Africa,RSA,400021443,1418.211807,1700.984519,34.0,0.360912,6.863084,0.103336,0.0,-0.103336,-1.000000,0.000000
2,Korea Republic,KOR,400021441,1612.547459,1481.485998,55.0,0.557826,13.772668,1.769696,2.0,0.230304,0.130138,0.145215
3,Czechia,CZE,400021441,1481.485998,1612.547459,41.0,0.341655,11.588785,0.636732,1.0,0.363268,0.570518,0.086290
4,Canada,CAN,400021449,1551.504089,1406.176640,67.0,0.528129,19.241495,1.494709,1.0,-0.494709,-0.330973,0.051971
5,Bosnia and Herzegovina,BIH,400021449,1406.176640,1551.504089,29.0,0.298916,4.765946,0.761922,1.0,0.238078,0.312471,0.209822
6,USA,USA,400021458,1688.534820,1488.048969,74.0,0.594737,21.856615,1.877016,4.0,2.122984,1.131042,0.183011
7,Paraguay,PAR,400021458,1488.048969,1688.534820,22.0,0.287417,6.794338,0.604615,1.0,0.395385,0.653944,0.147181
8,Qatar,QAT,400021447,1459.446908,1629.939343,18.0,0.287881,8.191307,0.518329,1.0,0.481671,0.929277,0.122081
9,Switzerland,SUI,400021447,1629.939343,1459.446908,79.0,0.587374,28.390107,3.141817,1.0,-2.141817,-0.681713,0.035224
